# Cross-Dataset Evaluation 2
## Train on BanglaSarc3-binary, test on Ben-Sarc-binary

This notebook trains **BanglaBERT** on the source dataset train/validation splits and evaluates on the target dataset test split.

- Source: `banglasarc3_binary`
- Target: `ben_sarc_binary`

In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch version:", torch.__version__)
try:
    import transformers
    print("Transformers version:", transformers.__version__)
except Exception as e:
    print("Could not read transformers version:", e)

Torch version: 2.11.0
Transformers version: 5.3.0


In [3]:
SPLITS = Path("../01_data/interim/splits")
TABLES = Path("../04_outputs/tables")
CHECKPOINTS = Path("../03_models/checkpoints")

TABLES.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01

source_dataset = "banglasarc3_binary"
target_dataset = "ben_sarc_binary"

In [4]:
train_df = pd.read_csv(SPLITS / f"{source_dataset}_train.csv")
val_df = pd.read_csv(SPLITS / f"{source_dataset}_val.csv")
test_df = pd.read_csv(SPLITS / f"{target_dataset}_test.csv")

print("Train source:", train_df.shape)
print("Val source:", val_df.shape)
print("Test target:", test_df.shape)
display(train_df.head())

Train source: (6413, 4)
Val source: (802, 4)
Test target: (2564, 4)


,text,label_binary,label_original,dataset_name
0,হুদাই কোনো দাবিই মানা হয়নি ২০ ওয়েভারের বাইরে...,0,Non-Sarcastic,banglasarc3
1,হে মুরুব্বি ভালো মেয়ের সন্ধান দিন করে ফেলি,0,Non-Sarcastic,banglasarc3
2,সফল কৃষক তাকে একটা নোবেল ছুঁড়ে মারা উচিৎ,0,Non-Sarcastic,banglasarc3
3,আমার পোলা এমন ছিলো না অই ছেমরির পাল্লায় পইরা এ...,1,Sarcastic,banglasarc3
4,নীলগাই কে ধরে দ্রুত আইনের আওতায় এনে উপযুক্ত শা...,1,Sarcastic,banglasarc3


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [6]:
train_df = train_df[["text", "label_binary"]].rename(columns={"label_binary": "label"})
val_df = val_df[["text", "label_binary"]].rename(columns={"label_binary": "label"})
test_df = test_df[["text", "label_binary"]].rename(columns={"label_binary": "label"})

In [7]:
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)

In [8]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

Map: 100%|██████████| 2564/2564 [00:00<00:00, 29767.17 examples/s]


In [9]:
train_ds = train_ds.remove_columns(["text"])
val_ds = val_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_binary": p_bin,
        "recall_binary": r_bin,
        "f1_binary": f1_bin,
        "macro_f1": f1_macro,
    }

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 48906.65it/s]
ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

In [12]:
training_args = TrainingArguments(
    output_dir=f"../03_models/checkpoints/cross_{source_dataset}_to_{target_dataset}",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [14]:
trainer.train()

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Binary,Recall Binary,F1 Binary,Macro F1
1,0.558492,0.504857,0.763092,0.734967,0.822943,0.776471,0.762241
2,0.407658,0.569692,0.764339,0.786486,0.725686,0.754864,0.763987


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.18it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.80it/s]
There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.Lay

TrainOutput(global_step=1604, training_loss=0.4830753393006741, metrics={'train_runtime': 921.5458, 'train_samples_per_second': 13.918, 'train_steps_per_second': 1.741, 'total_flos': 843665599011840.0, 'train_loss': 0.4830753393006741, 'epoch': 2.0})

In [15]:
test_output = trainer.predict(test_ds)
test_preds = np.argmax(test_output.predictions, axis=-1)
test_labels = np.array(test_df["label"])

acc = accuracy_score(test_labels, test_preds)
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    test_labels, test_preds, average="macro", zero_division=0
)
p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
    test_labels, test_preds, average="binary", zero_division=0
)
cm = confusion_matrix(test_labels, test_preds)

print("Cross-dataset Test Accuracy:", round(acc, 4))
print("Cross-dataset Test Precision (binary):", round(p_bin, 4))
print("Cross-dataset Test Recall (binary):", round(r_bin, 4))
print("Cross-dataset Test F1 (binary):", round(f1_bin, 4))
print("Cross-dataset Test Macro-F1:", round(f1_macro, 4))
print("\nConfusion Matrix:")
print(cm)

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Cross-dataset Test Accuracy: 0.6853
Cross-dataset Test Precision (binary): 0.6745
Cross-dataset Test Recall (binary): 0.7161
Cross-dataset Test F1 (binary): 0.6947
Cross-dataset Test Macro-F1: 0.685

Confusion Matrix:
[[839 443]
 [364 918]]


In [16]:
print(classification_report(test_labels, test_preds, zero_division=0))

              precision    recall  f1-score   support

           0       0.70      0.65      0.68      1282
           1       0.67      0.72      0.69      1282

    accuracy                           0.69      2564
   macro avg       0.69      0.69      0.68      2564
weighted avg       0.69      0.69      0.68      2564



In [17]:
results = [
    {
        "model": "banglabert_cross_dataset",
        "source_dataset": source_dataset,
        "target_dataset": target_dataset,
        "accuracy": acc,
        "precision_binary": p_bin,
        "recall_binary": r_bin,
        "f1_binary": f1_bin,
        "macro_f1": f1_macro,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "max_length": MAX_LENGTH,
        "seed": SEED,
    }
]

results_df = pd.DataFrame(results)
results_df.to_csv(TABLES / f"cross_{source_dataset}_to_{target_dataset}_results.csv", index=False)

with open(TABLES / f"cross_{source_dataset}_to_{target_dataset}_confusion_matrix.json", "w", encoding="utf-8") as f:
    json.dump({"confusion_matrix": cm.tolist()}, f, ensure_ascii=False, indent=2)

results_df

,model,source_dataset,target_dataset,accuracy,precision_binary,recall_binary,f1_binary,macro_f1,epochs,batch_size,learning_rate,max_length,seed
0,banglabert_cross_dataset,banglasarc3_binary,ben_sarc_binary,0.685257,0.674504,0.716069,0.694665,0.684958,2,8,0.00002,128,42
